 الخطة الكاملة لـ Python Notebook (The Python Pipeline)هاد الـ Pipeline كينقسم لـ 5 ديال الخطوات رئيسية، وكل خطوة عندها الهدف والأدوات ديالها
 :1. Data Ingestion & Initial Inspection (الاستكشاف الأولي)شنو غنديرو: نقراو الداتا ونطلوا عليها نشوفوا واش تحمّلات مقادة.الأدوات: pandas (pd.read_csv, .head(), .shape, .info(), .describe()).النتيجة: نعرفوا عدد السطور/الأعمدة، وأسماء الفاريابل، وأشمن أعمدة فيهم Nulls.

 2. Data Cleaning & Type Casting (التنظيف وتصحيح الأنواع)شنو غنديرو: تنقية البيانات من الأخطاء باش تولي جاهزة للتحليل.الأدوات: pandas (.fillna(), .dropna(), .drop_duplicates(), pd.to_datetime()).العمليات:معالجة الـ Missing Values فـ reviews_per_month و name.تحويل last_review لـ صيغة تاريخ (datetime).التأكد بلي price و minimum_nights أرقام منطقية (تحييد الـ Outliers أو الأخطاء بحال ثمن = 0$).

 3. Feature Engineering & Business Metrics (خلق فاريابل جداد)شنو غنديرو: نزيدو أعمدة حسوبية جديدة كتحل مشاكل Business اللي طلب المستثمر.الأدوات: Python Custom Functions, Vectorized operations.العمليات:خلق عمود estimated_revenue ($price \times minimum\_nights \times number\_of\_reviews$).استخراج الكلمات المفتاحية فـ العنوان name (مثلا: خلق عمود is_luxury واش فيه كلمة Luxury/Cozy).

 4. Exploratory Data Analysis - EDA (الاستكشاف الإحصائي والتحليلي)شنو غنديرو: نجاوبو على الـ 6 العوامل اللي حددنا فـ Ask Phase بـ أرقام وغرافيكس خفاف.الأدوات: pandas (.groupby(), .value_counts(), .corr()), seaborn, matplotlib.العمليات:Pricing & Revenue: متوسط الأسعار حسب الـ Boroughs (Manhattan, Brooklyn...).Demand & Quality: الأحياء اللي فيهم أعلى طلب وتواجد الـ Hidden Gems.Room Types & Host Profiles: نسبة كل نوع سكن ومنافسة المستثمرين الكبار.

 5. Data Export for SQL (تصدير البيانات النظيفة)شنو غنديرو: نخرجوا ملف CSV منظف ومقاد 100% باش نهزوه ونطلعوه لـ MySQL Workbench.الأدوات: df.to_csv('AB_NYC_2019_Cleaned.csv').

In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("/content/AB_NYC_2019.csv") # read_csv هي اللي كتقرا وتصاوب DataFrame وتسجلو فـ متغير سميتو df

In [3]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [4]:
df.shape

(48895, 16)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

In [6]:
df.isnull().sum()

,0
id,0
name,16
host_id,0
host_name,21
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


الاستراتيجية الأولى: Data Imputation (التعويض المنطقي)

متى كنزيدوها: فاش كيكون الخلل عنده تفسير رياضي ولا Business واضح.

تطبيقها فـ البروجي ديالنا:

عمود reviews_per_month فيه 10,052 قيمة مفقودة. السبب ماشي خطأ، بل حيت هاد الشقق جداد أو ما عمر حتى حد حجزهم (0 التقييمات).

القرار: نعوضو القيم المفقودة فـ هاد العمود بـ 0 (fillna(0)). هكا الداتا كتولي منطقية 100% وما كنزيدوش أرقام وهمية.

الاستراتيجية الثانية: Placeholder Categorization (الترميز النصي)

متى كنزيدوها: فاش كيكون العمود نصي (Text/String) وما محتاجينش نمسحو السطر كامل باش ما نضيعوش المعطيات الماليه والأرقام الأخرى (بحال السعر والموقع).

تطبيقها فـ البروجي ديالنا:

name (ناقصين 16) و host_name (ناقصين 21).

القرار: نعوضوهم بـ نص افتراضي بحال "Unknown" أو "No Name". هكا كنحافظو على السطر والبيانات الاستثمارية اللي فيه بلا ما نضيعوها.

الاستراتيجية الثالثة: Business Logic & Data Type Conversion (المنطق الجغرافي والزمني)

تطبيقها فـ البروجي ديالنا:

last_review فيه تاريخ (Date)، والسطور الخاوية كتعني "ما كاين حتى تقييم سابق". فـ التنظيف كنزيدو نخليوه خاوي ولا نعوضوه بـ تاريخ وهمي إذا طلب السيستم، ولكن الأهم كتحولو النوع ديالو لـ datetime.

الاستراتيجية الرابعة: Deletion (مسح السطور/الأعمدة - Drop)

متى كنزيدوها: فقط فاش كيكون العمود فيه أكثر من 50%-60% خاوي وما عندوش أهمية فـ التحليل، أو السطر خاوي كامل. (فـ البروجي ديالنا ما محتاجينش نمسحو حيت النسبة قليلة وممكن تعويضها).

In [7]:
df['name'] = df['name'].fillna("no name")
df['host_name'] = df['host_name'].fillna("unkown")
df['last_review'] = pd.to_datetime(df['last_review'])
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)


In [8]:
df.isnull().sum()

,0
id,0
name,0
host_id,0
host_name,0
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


In [9]:
df = df.dropna(subset=['room_type','price','minimum_nights','number_of_reviews','calculated_host_listings_count','availability_365'])

In [10]:
df.isnull().sum()

,0
id,0
name,0
host_id,0
host_name,0
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


In [11]:
#Feature Engineering & Business Metrics challenge 3  نزيدو أعمدة حسوبية جديدة فـ التابلو حيت المعطيات الخام ما كافياش باش نجاوبو على الأسئلة ديال الاستثما

In [12]:
#estimated_revenue

In [13]:
df['estimated_revenue'] = df['price']*df['minimum_nights']*df['number_of_reviews']

In [14]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,estimated_revenue
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365,1341
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355,10125
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaT,0.00,1,365,0
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194,24030
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0,7200


In [15]:
# is_luxury impacts the price

In [16]:
df['is_luxury']= df['name'].str.contains('luxury',case=False)# case pour ignorer la sensibilité de la casse

In [17]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,estimated_revenue,is_luxury
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365,1341,False
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355,10125,False
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaT,0.00,1,365,0,False
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194,24030,False
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0,7200,False


In [18]:
luxury_appartment = df[df['is_luxury'] == True]
luxury_appartment = df.query('is_luxury == True')

In [19]:
luxury_appartment[['name','price','is_luxury']].head()

,name,price,is_luxury
108,"1,800 sq foot in luxury building",100,True
158,Luxury Brownstone in Boerum Hill,475,True
174,Financial District Luxury Loft,196,True
183,Bright Spacious Luxury Condo,200,True
201,3 floors of luxury!,265,True


***EDA:***

1. Pricing & Yield (الأسعار والأرباح حسب المناطق):

السؤال: اشنو هو متوسط السعر والمدخول التقديري فـ كل منطقة (neighbourhood_group) فـ نيويورك؟

شنو غنحتاجو فـ Python: غنحتاجو groupby('neighbourhood_group') ونحسبو الـ .mean() لـ price و estimated_revenue.

2. Demand & Activity (الطلب والنشاط):

السؤال: أشنو هما الأحياء اللي فيهم أعلى طلب من الكليان؟

شنو غنحتاجو فـ Python: نفلترو ونجمعو حسب neighbourhood ونشوفو الأعمدة ديال number_of_reviews و availability_365.

المتوسط (.mean()): حساس بزاف للقيم المتطرفة (Outliers) حيت كيجمع كلشي ويقسمه. شقة واحدة خيالية تقدر تطلع الحي كامل للسماء.

الوسيط (.median()): كياخد الرقم اللي فـ النص بالضبط. كيعطيك الصورة ديال "الشقة العادية فـ الحي شحال كتدخل" بلا ما يتأثر بـ الشواذ.

3. Quality & Hidden Gems (الفرص الاستثمارية المخفية):

السؤال: واش كاينين أحياء الطلب عليهم طالع والتقييمات عالية ولكن الأسعار ديال الشراء/الكراء فيها مناسبة؟

شنو غنحتاجو فـ Python: غنحتاجو نقارنو متوسط الأسعار (price) مقابل النشاط (reviews_per_month) حسب كل حي.

4. Property Types (أنواع الشقق والأرباح):

السؤال: اشنو هو نوع الغرف (Entire home ولا Private room) اللي كيدخل أعلى مدخول تقديري؟

شنو غنحتاجو فـ Python: نديرو groupby('room_type') ونحسبو متوسط الـ estimated_revenue.

5. Competition & Host Profile (المنافسة ونوع المستثمرين):

السؤال: واش السوق مسيطرين عليه مستثمرين كبار عندهم بزاف ديال الشقق، ولا غير أفراد عاديين عندهم شقة وحدة؟

شنو غنحتاجو فـ Python: نلخصو المعطيات ديال العمود calculated_host_listings_count ونحسبو النسبة المئوية ديال كل فئة.

6. Marketing Impact (تأثير التسويق فـ العنوان):

السؤال: واش الشقق اللي فـ عنوانها كلمات بحال "Luxury" ثمنها وطرق الربح فيها طالعين بالمقارنة مع العادية؟

شنو غنحتاجو فـ Python: نقارنو المعدل ديال price بين الشقق اللي فيها is_luxury == True والاقل منها is_luxury == False.

In [20]:
# 1. الأسعار والأرباح حسب المناطق

grouped = df.groupby('neighbourhood_group')
avg_price = grouped[['price','estimated_revenue']].mean().sort_values(by='price',ascending=False)

#avg_estimated_revenue  = grouped['estimated_revenue'].mean()

In [21]:
print('avg_price')
print(avg_price)


avg_price
                          price  estimated_revenue
neighbourhood_group                               
Manhattan            196.875814       16476.863811
Brooklyn             124.383207       12169.413002
Staten Island        114.812332        6929.096515
Queens                99.517649        7155.884045
Bronx                 87.496792        4968.092576


In [22]:
#print('avg_estimated_revenue')
#print(avg_estimated_revenue)

In [23]:
 # 2. we will need median of number_of_reviews, reviews_per_month, availability_365 الطلب والنشاط

grouped = df.groupby('neighbourhood')
top_demand_neighborhoods = grouped[['number_of_reviews','reviews_per_month','availability_365']].median()
top_demand_neighborhoods = top_demand_neighborhoods.sort_values(by='number_of_reviews',ascending=False).head(10)
print(top_demand_neighborhoods)

                 number_of_reviews  reviews_per_month  availability_365
neighbourhood                                                          
Silver Lake                  118.5              4.340             162.0
Eltingville                   83.0              2.380             258.0
Richmondtown                  79.0              2.560             300.0
Manhattan Beach               50.0              1.170             242.0
East Morrisania               40.5              2.795             149.0
Lighthouse Hill               39.0              1.165             215.5
Pelham Gardens                38.5              2.035             126.5
Dyker Heights                 37.0              1.160             138.0
Graniteville                  36.0              1.010             226.0
Highbridge                    32.0              1.450             134.0


118.5: عدد التعاليق الكلية اللي تكتدبات على الشقة ملي تحلات (كتعطينا العمر والتاريخ ديال الشقة).

4.34: عدد التعاليق الجديدة اللي كتوصل الشقة كل شهر (كتعطينا السرعة والحركة ديال الحجز فـ الوقت الحالي).

In [24]:
# To evaluate if a neighborhood's availability and reviews are good or bad,
# we need to compare them against NYC's overall median values (Benchmark).

availibility_NYC_overall_median = df['availability_365'].median()
number_of_reviews_NYC_overall_median = df['number_of_reviews'].median()

print('availibility_NYC_overall_median')
print(availibility_NYC_overall_median)
print('number_of_reviews_NYC_overall_median')
print(number_of_reviews_NYC_overall_median)

availibility_NYC_overall_median
45.0
number_of_reviews_NYC_overall_median
5.0


In [25]:
# Affordable & High-Demand Neighborhoods
#نلقاو أحياء فيها أعلى عائد على الاستثمار (High ROI)
# (Return on Investment) كيعني: شحال غنربح من كل درهم ولا دولار حطيتو فـ المشروع.
#كنعطيوا للمستثمر خريطة بـ الأحياء المخفية اللي ما فيهاش منافسة شرسة فـ السعر ولكن فيها كليان ديما حاضرين.

In [26]:
nyc_price_median = df['price'].median()
nyc_activity_median  = df['number_of_reviews'].median()

grouped = df.groupby('neighbourhood')
median_price_numofreviews =grouped[['price','number_of_reviews']].median()
median_price_numofreviews = median_price_numofreviews.sort_values(by='price',ascending=False)

print(median_price_numofreviews)

                price  number_of_reviews
neighbourhood                           
Fort Wadsworth  800.0                0.0
Woodrow         700.0                0.0
Tribeca         295.0                2.0
Neponsit        274.0                7.0
NoHo            250.0                3.5
...               ...                ...
Corona           40.0               16.0
New Dorp Beach   40.0                0.0
Hunts Point      40.0                7.0
Castle Hill      39.0                0.0
Concord          34.5               24.5

[221 rows x 2 columns]


النتائج اللي طالعين عندك فـ الشاشة كيعطيونا 3 ديال الـ Business Insights قاصحين وواضحين بزاف:

الأحياء الغالية بـ إقبال ضعيف (High Price, Low Demand):

أحياء بحال Fort Wadsworth (800$) و Woodrow (700$) عندهم السعر طالع بزاف، ولكن وسيط التقييمات 0.0! هاد الأحياء غالباً فيهم شقق Luxury ولا كبار، ولكن ما كيتكراوش فـ Airbnb وميتين فـ الطلب.

الأحياء الرخيصة بـ إقبال عالي (Low Price, High Demand - الفرص الاستثمارية):

شوف الفرق فـ الأسفل: حي Concord السعر ديالو غير 34.5$ والنشاط ديال التقييمات طالع لـ 24.5!

وحي Corona السعر ديالو 40$ والنشاط 16.0 (أعلى بكثير من معدل نيويورك اللي هو 5).

الأحياء الميتة (Low Price, Zero Demand):

أحياء بحال Castle Hill (39$) و New Dorp Beach (40$) رخاص ولكن التقييمات 0.0، يعني وخا رخاص الكليان ما كيمشيوش ليهم.

In [27]:
best_to_choose = median_price_numofreviews[
    (median_price_numofreviews['price']< nyc_price_median )&
    (median_price_numofreviews['number_of_reviews']>nyc_activity_median)
]

In [28]:
#4. Impact of Property Type:

#السؤال: How does the room type (Entire home vs. Private room) affect listing price and demand?

#الهدف: نعرفوا شنو هو نوع العقار الإستراتيجي اللي خاص المستثمر يركز عليه فـ الشراء ولا الكراء باش يحقق أعلى مدخول.

In [29]:
grouped_room = df.groupby('room_type')

# Utilisation des crochets [] pour accéder aux colonnes
avg_price_by_room = grouped_room['price'].median()
avg_estimated_revenue_by_room = grouped_room['estimated_revenue'].median()
avg_num_of_rev = grouped_room['number_of_reviews'].median()

print('avg_price_by_room')
print(avg_price_by_room)
print('\n avg_estimated_revenue_by_room')
print(avg_estimated_revenue_by_room)
print('\n avg_num_of_rev')
print(avg_num_of_rev)


avg_price_by_room
room_type
Entire home/apt    160.0
Private room        70.0
Shared room         45.0
Name: price, dtype: float64

 avg_estimated_revenue_by_room
room_type
Entire home/apt    3240.0
Private room        900.0
Shared room         340.0
Name: estimated_revenue, dtype: float64

 avg_num_of_rev
room_type
Entire home/apt    5.0
Private room       5.0
Shared room        4.0
Name: number_of_reviews, dtype: float64


للمستثمر: الاستثمار فـ Entire home/apt هو الخيار الأفضل بدون منازع للوصول لأعلى عائد مالي، حيت هاد القطاع كيحافظ على نفس الإقبال ديال الغرف الخاصة ولكن بـ هامش ربح وسعر مضاعف.

In [30]:
# combien d'partement possede ce host c calculated_host_listings_count
# Competition & Host Profile:** Is the market dominated by single hosts or multi-listing real estate investors?
# to know the top investors   كيبيّن قوة الحيتان الكبار فـ السوق.

multi_host = df[df['calculated_host_listings_count']>3]

top_5_investors = multi_host[['host_id','host_name','calculated_host_listings_count']].drop_duplicates().sort_values(by='calculated_host_listings_count',ascending=False).head(5)
print(top_5_investors)

# if we want Percentage of Commercial Listings (>3 properties):
Percentage_of_Commercial_Listings = (len(multi_host)/len(df))*100
print(f'Percentage of Commercial Listings (>3 properties): {Percentage_of_Commercial_Listings:.2f}%')




         host_id       host_name  calculated_host_listings_count
38293  219517861    Sonder (NYC)                             327
26137  107434423      Blueground                             232
9740    30283594            Kara                             121
32718  137358866          Kazuya                             103
5093    16098958  Jeremy & Laura                              96
Percentage of Commercial Listings (>3 properties): 14.48%


ورغم وجود حيتان عقارية تسيطر على مئات الإدراجات، إلا أن المستثمرين الكبار يمثلون 14% فقط من إجمالي السوق، مما يعني أن السوق لا يزال يتيح فرصاً واعدة للمستثمرين الجدد دون الخوف من الاحتكار الكامل.

In [31]:
#**6. Marketing Impact:** Do specific listing title keywords (e.g., "Luxury", "Cozy", "Modern") correlate with higher prices?

keywords = ['luxury', 'Modern', 'Cozy', 'Spacious', 'Charming', 'Renovated']

res = []


for k in keywords:
  app_containing = df[df['name'].str.contains(k,case=False, na=False)]
  res.append({
      'keyword':k,
      'Median Price':app_containing['price'].median(),
      'Median num_rev':app_containing['number_of_reviews'].median(),
      'Listing Count':len(app_containing)
  })
# res is a list and it doens't have the sort function that is why we should transform it into a dataframe

Keyword_Marketing_Impact_Comparison = pd.DataFrame(res).sort_values(by='Median Price',ascending=False)

print(Keyword_Marketing_Impact_Comparison)


     keyword  Median Price  Median num_rev  Listing Count
0     luxury         195.0             3.0           1718
1     Modern         140.0             6.0           1838
4   Charming         125.0             6.0           1388
5  Renovated         115.0             7.0            700
3   Spacious         110.0             5.0           3800
2       Cozy          85.0             7.0           5111


كلمة "Luxury" هي الملكة فـ السعر (Price Premium):

كتحقق أعلى وسيط سعر بـ $195 (أكثر من ضعف سعر Cozy)، ولكن الإقبال عليها قلّ شوية (3 reviews)، هادشي منطقي حيت المستهلك اللي كيكري الفخامة كيكون فئة خاصة.

استراتيجية التوازن الذكي (Modern & Charming):

الكلمات بحال Modern ($140) و Charming ($125) كيحققوا توازن رائع بين سعر مرتفع وإقبال عالي (6 reviews).

كلمة "Cozy" هي محرك الحجوزات (High Occupancy Driver):

أكثر كلمة مستعملة فـ نيويورك (5,111 عقار) بـ سعر اقتصادي ($85) وأعلى نسبة إقبال (7 reviews).



In [32]:
from numpy import False_
df.to_csv('AB_NYC_2019_Cleaned.csv',index=False)